# Dịch Dữ Liệu Job từ Tiếng Việt sang Tiếng Anh

**Các trường được dịch:** description, requirements_text, education_level, level, job_type, benefits, required_skills

**Lưu ý:** Chạy từng cell theo thứ tự

## 1. Cài đặt thư viện

In [66]:
!pip install pymysql sqlalchemy deep-translator langdetect pandas tqdm cryptography

## 2. Import thư viện

In [67]:
import pymysql
import pandas as pd
import json
import time
import ssl
import tempfile
import os
from sqlalchemy import create_engine, text
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException
from tqdm.notebook import tqdm
from typing import Optional, List
import warnings
warnings.filterwarnings('ignore')
print('✅ Import OK')

✅ Import OK


## 3. Cấu hình Database và SSL Certificate

**QUAN TRỌNG:** 
- Thay đổi thông tin database
- Copy nội dung file .pem và paste vào CA_CERT_CONTENT

In [68]:
# ========== CẤU HÌNH DATABASE ==========
DB_CONFIG = {
    'host': 'gateway01.ap-southeast-1.prod.aws.tidbcloud.com',
    'port': 4000,
    'user': '4GJhpnEevqoZfyD.root',
    'password': 'oiK2dgnVVJLVHL4v',
    'database': 'data-mining',
    'charset': 'utf8mb4'
}

# ========== PASTE NỘI DUNG CERTIFICATE ==========
# Mở file .pem bằng Notepad, copy toàn bộ và paste vào đây
CA_CERT_CONTENT = """
-----BEGIN CERTIFICATE-----
MIIFazCCA1OgAwIBAgIRAIIQz7DSQONZRGPgu2OCiwAwDQYJKoZIhvcNAQELBQAw
TzELMAkGA1UEBhMCVVMxKTAnBgNVBAoTIEludGVybmV0IFNlY3VyaXR5IFJlc2Vh
cmNoIEdyb3VwMRUwEwYDVQQDEwxJU1JHIFJvb3QgWDEwHhcNMTUwNjA0MTEwNDM4
WhcNMzUwNjA0MTEwNDM4WjBPMQswCQYDVQQGEwJVUzEpMCcGA1UEChMgSW50ZXJu
ZXQgU2VjdXJpdHkgUmVzZWFyY2ggR3JvdXAxFTATBgNVBAMTDElTUkcgUm9vdCBY
MTCCAiIwDQYJKoZIhvcNAQEBBQADggIPADCCAgoCggIBAK3oJHP0FDfzm54rVygc
h77ct984kIxuPOZXoHj3dcKi/vVqbvYATyjb3miGbESTtrFj/RQSa78f0uoxmyF+
0TM8ukj13Xnfs7j/EvEhmkvBioZxaUpmZmyPfjxwv60pIgbz5MDmgK7iS4+3mX6U
A5/TR5d8mUgjU+g4rk8Kb4Mu0UlXjIB0ttov0DiNewNwIRt18jA8+o+u3dpjq+sW
T8KOEUt+zwvo/7V3LvSye0rgTBIlDHCNAymg4VMk7BPZ7hm/ELNKjD+Jo2FR3qyH
B5T0Y3HsLuJvW5iB4YlcNHlsdu87kGJ55tukmi8mxdAQ4Q7e2RCOFvu396j3x+UC
B5iPNgiV5+I3lg02dZ77DnKxHZu8A/lJBdiB3QW0KtZB6awBdpUKD9jf1b0SHzUv
KBds0pjBqAlkd25HN7rOrFleaJ1/ctaJxQZBKT5ZPt0m9STJEadao0xAH0ahmbWn
OlFuhjuefXKnEgV4We0+UXgVCwOPjdAvBbI+e0ocS3MFEvzG6uBQE3xDk3SzynTn
jh8BCNAw1FtxNrQHusEwMFxIt4I7mKZ9YIqioymCzLq9gwQbooMDQaHWBfEbwrbw
qHyGO0aoSCqI3Haadr8faqU9GY/rOPNk3sgrDQoo//fb4hVC1CLQJ13hef4Y53CI
rU7m2Ys6xt0nUW7/vGT1M0NPAgMBAAGjQjBAMA4GA1UdDwEB/wQEAwIBBjAPBgNV
HRMBAf8EBTADAQH/MB0GA1UdDgQWBBR5tFnme7bl5AFzgAiIyBpY9umbbjANBgkq
hkiG9w0BAQsFAAOCAgEAVR9YqbyyqFDQDLHYGmkgJykIrGF1XIpu+ILlaS/V9lZL
ubhzEFnTIZd+50xx+7LSYK05qAvqFyFWhfFQDlnrzuBZ6brJFe+GnY+EgPbk6ZGQ
3BebYhtF8GaV0nxvwuo77x/Py9auJ/GpsMiu/X1+mvoiBOv/2X/qkSsisRcOj/KK
NFtY2PwByVS5uCbMiogziUwthDyC3+6WVwW6LLv3xLfHTjuCvjHIInNzktHCgKQ5
ORAzI4JMPJ+GslWYHb4phowim57iaztXOoJwTdwJx4nLCgdNbOhdjsnvzqvHu7Ur
TkXWStAmzOVyyghqpZXjFaH3pO3JLF+l+/+sKAIuvtd7u+Nxe5AW0wdeRlN8NwdC
jNPElpzVmbUq4JUagEiuTDkHzsxHpFKVK7q4+63SM1N95R1NbdWhscdCb+ZAJzVc
oyi3B43njTOQ5yOf+1CceWxG1bQVs5ZufpsMljq4Ui0/1lvh+wjChP4kqKOJ2qxq
4RgqsahDYVvTH9w7jXbyLeiNdd8XM2w9U/t7y0Ff/9yi0GE44Za4rF2LN9d11TPA
mRGunUHBcnWEvgJBQl9nJEiU0Zsnvgc/ubhPgXRR4Xq37Z0j4r7g1SgEEzwxA57d
emyPxgcYxn/eR44/KJ4EBs+lVDR3veyJm+kXQ99b21/+jh5Xos1AnX5iItreGCc=
-----END CERTIFICATE-----
""".strip()

CLIENT_CERT_CONTENT = None
CLIENT_KEY_CONTENT = None

print(f'📊 Database: {DB_CONFIG["database"]}')
print(f'🔗 Host: {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')
print(f'📜 CA Cert: {"✅" if len(CA_CERT_CONTENT) > 100 else "❌ Chưa config"}')

📊 Database: data-mining
🔗 Host: gateway01.ap-southeast-1.prod.aws.tidbcloud.com:4000
📜 CA Cert: ✅


## 4. Tạo SSL Connection

In [69]:
temp_files = []
try:
    ssl_context = ssl.create_default_context()
    
    if CA_CERT_CONTENT:
        ca_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        ca_temp.write(CA_CERT_CONTENT)
        ca_temp.close()
        temp_files.append(ca_temp.name)
        ssl_context.load_verify_locations(ca_temp.name)
        print('✅ Loaded CA certificate')
    
    if CLIENT_CERT_CONTENT and CLIENT_KEY_CONTENT:
        cert_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        cert_temp.write(CLIENT_CERT_CONTENT)
        cert_temp.close()
        key_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        key_temp.write(CLIENT_KEY_CONTENT)
        key_temp.close()
        temp_files.extend([cert_temp.name, key_temp.name])
        ssl_context.load_cert_chain(cert_temp.name, key_temp.name)
        print('✅ Loaded client certificate')
    
    DATABASE_URL = f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}?charset={DB_CONFIG['charset']}"
    engine = create_engine(DATABASE_URL, connect_args={'ssl': ssl_context}, pool_pre_ping=True)
    
    print('\n🔄 Testing connection...')
    with engine.connect() as conn:
        result = conn.execute(text('SELECT COUNT(*) FROM jobs'))
        total = result.fetchone()[0]
        print(f'\n✅ Kết nối thành công!')
        print(f'📊 Tổng số jobs: {total:,}')
        try:
            ssl_status = conn.execute(text("SHOW STATUS LIKE 'Ssl_cipher'"))
            cipher = ssl_status.fetchone()
            if cipher and cipher[1]: print(f'🔒 SSL Cipher: {cipher[1]}')
        except: pass
except Exception as e:
    print(f'❌ Lỗi: {e}')
    for f in temp_files:
        try: os.unlink(f)
        except: pass
    raise

✅ Loaded CA certificate

🔄 Testing connection...

✅ Kết nối thành công!
📊 Tổng số jobs: 2,985
🔒 SSL Cipher: TLS_AES_128_GCM_SHA256


## 5. Định nghĩa hàm dịch

In [70]:
class TranslationHelper:
    def __init__(self):
        self.translator = GoogleTranslator(source='auto', target='en')
        self.cache = {}
        self.stats = {
            'translated': 0, 
            'skipped_empty': 0,
            'skipped_short': 0,
            'errors': 0,
            'cached': 0,
            'already_english': 0
        }
    
    def should_translate(self, text):
        """Simple check - translate everything except empty, very short, or numbers"""
        if not text or not str(text).strip():
            return False
        
        text_str = str(text).strip()
        
        # Skip very short text (1-2 chars) and pure numbers
        if len(text_str) <= 2 or text_str.replace('.', '').replace(',', '').isdigit():
            return False
        
        return True
    
    def translate_text(self, text, field_name='', max_retries=3):
        """Translate ALL text to English - NO language detection, just translate"""
        if not text or not str(text).strip():
            self.stats['skipped_empty'] += 1
            return text
        
        text_str = str(text).strip()
        
        # Skip very short text and pure numbers
        if len(text_str) <= 2:
            self.stats['skipped_short'] += 1
            return text_str
        
        if text_str.replace('.', '').replace(',', '').replace('-', '').isdigit():
            self.stats['skipped_short'] += 1
            return text_str
        
        # Check cache first
        if text_str in self.cache:
            self.stats['cached'] += 1
            return self.cache[text_str]
        
        # Translate with retry - NO language detection, just translate everything
        for attempt in range(max_retries):
            try:
                if len(text_str) > 4500:
                    # Split long text into chunks
                    chunks = [text_str[i:i+4500] for i in range(0, len(text_str), 4500)]
                    translated_chunks = []
                    for chunk in chunks:
                        result = self.translator.translate(chunk)
                        translated_chunks.append(result if result else chunk)
                    result = ' '.join(translated_chunks)
                else:
                    result = self.translator.translate(text_str)
                
                # Cache the result
                if result and result.strip():
                    # Check if translation actually changed anything
                    if result.strip().lower() == text_str.lower():
                        self.stats['already_english'] += 1
                    else:
                        self.stats['translated'] += 1
                    
                    self.cache[text_str] = result
                    time.sleep(0.05)  # Rate limiting
                    return result
                else:
                    # Translation failed, return original
                    self.cache[text_str] = text_str
                    return text_str
                
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(1 * (attempt + 1))
                else:
                    if attempt == 0 or len(self.stats) < 5:  # Show first few errors
                        print(f'⚠️ Translation error for {field_name}: {str(e)[:50]}')
                    self.stats['errors'] += 1
                    return text_str
        
        return text_str
    
    def translate_list(self, items, field_name=''):
        """Translate ALL items in list - NO filtering"""
        if not items or not isinstance(items, list): 
            return items
        
        result = []
        for item in items:
            if isinstance(item, str):
                translated = self.translate_text(item, field_name)
                result.append(translated)
            else:
                result.append(item)
        return result
    
    def print_stats(self):
        """Print translation statistics"""
        print(f"📊 Translated: {self.stats['translated']:,} | "
              f"Already English: {self.stats['already_english']:,} | "
              f"Cached: {self.stats['cached']:,} | "
              f"Skipped: {self.stats['skipped_empty'] + self.stats['skipped_short']:,} | "
              f"Errors: {self.stats['errors']:,}")
    
    def get_summary(self):
        """Get detailed summary"""
        return {
            'total_operations': sum(self.stats.values()),
            'cache_size': len(self.cache),
            'stats': self.stats
        }

print('✅ TranslationHelper class defined (MAXIMUM AGGRESSIVE mode)')
print('🔥 Strategy: Translate EVERYTHING to English (no language detection)')
print('⚡ Only skips: empty text, 1-2 chars, pure numbers')
print('💪 All Vietnamese data WILL be translated')

✅ TranslationHelper class defined (MAXIMUM AGGRESSIVE mode)
🔥 Strategy: Translate EVERYTHING to English (no language detection)
⚡ Only skips: empty text, 1-2 chars, pure numbers
💪 All Vietnamese data WILL be translated


## 6. Lấy dữ liệu từ Database

In [71]:
query = '''
SELECT * FROM jobs ORDER BY id LIMIT 1
'''

print('📥 Loading data and analyzing columns...')
df = pd.read_sql(query, engine)

# Get all columns
all_columns = df.columns.tolist()
print(f'\n✅ Found {len(all_columns)} columns in jobs table:\n')

# Show all columns with sample data
print('='*100)
print(f'{"Column Name":<30} | {"Data Type":<15} | {"Sample Value":<50}')
print('='*100)

for col in all_columns:
    dtype = str(df[col].dtype)
    sample = str(df[col].iloc[0])[:50] if pd.notna(df[col].iloc[0]) else 'NULL'
    print(f'{col:<30} | {dtype:<15} | {sample}')

print('='*100)

# Now load ALL data (removed WHERE is_active = 1 filter)
print('\n📥 Loading ALL jobs from database...')
print('🔥 IMPORTANT: Translating ALL jobs, not just active ones!')
query_all = 'SELECT * FROM jobs ORDER BY id'
df = pd.read_sql(query_all, engine)
print(f'✅ Loaded {len(df):,} jobs with {len(df.columns)} columns')

# Show breakdown of active vs inactive
if 'is_active' in df.columns:
    active_count = df[df['is_active'] == 1].shape[0] if 'is_active' in df.columns else 0
    inactive_count = len(df) - active_count
    print(f'   📊 Active jobs (is_active=1): {active_count:,}')
    print(f'   📊 Inactive/Other jobs: {inactive_count:,}')

df.head()

📥 Loading data and analyzing columns...

✅ Found 27 columns in jobs table:

Column Name                    | Data Type       | Sample Value                                      
id                             | int64           | 3462
title                          | object          | Frontend developer (PA project)
company_name                   | object          | CUBICASA
location                       | object          | Quận 1, Hồ Chí Minh
source                         | object          | topdev
source_url                     | object          | https://topdev.vn/detail-jobs/frontend-developer-p
description                    | object          | Your role & responsibilities:
We’re looking for a

requirements_text              | object          | Your skills & qualifications:

Experience in front
job_type                       | object          | OTHER
level                          | object          | NULL
salary_min                     | object          | NULL
salary_max         

,id,title,company_name,location,source,source_url,description,requirements_text,job_type,level,...,certifications,required_skills,preferred_skills,benefits,company_rating,is_remote,is_active,created_at,updated_at,crawled_at
0,3462,Frontend developer (PA project),CUBICASA,"Quận 1, Hồ Chí Minh",topdev,https://topdev.vn/detail-jobs/frontend-develop...,Your role & responsibilities:\nWe’re looking f...,Your skills & qualifications:\n\nExperience in...,OTHER,None,...,None,"[""CSS"", ""HTML"", ""Git"", ""Restful Api"", ""SASS"", ...",None,"[""13th month bonusParticipate in large-scale a...",None,NaN,1.0,2026-01-11 15:11:56,2026-01-17 17:16:15,2026-01-11 15:11:56
1,3463,QA Engineer (Automotive),42dot Vietnam,"Thành phố Hồ Chí Minh, Hồ Chí Minh",topdev,https://topdev.vn/detail-jobs/qa-engineer-auto...,None,None,OTHER,None,...,None,"[""QA"", ""Tester"", ""Unit Testing"", ""Integration ...",None,"[""Premium office in District 1, free parking, ...",None,NaN,1.0,2026-01-11 15:12:00,2026-01-17 17:16:15,2026-01-11 15:12:00
2,3464,Java Developer,WEEDS VINA,"Quận Đống Đa, Hà Nội",topdev,https://topdev.vn/detail-jobs/java-developer-w...,None,None,OTHER,None,...,None,"[""Java"", ""JavaScript"", ""JQuery"", ""VueJS"", ""Rea...",None,"[""Attractive salary, 13th month salary, other ...",None,NaN,1.0,2026-01-11 15:12:04,2026-01-17 17:16:15,2026-01-11 15:12:04
3,3465,Nhân viên chế bản - Webtoon/ Manga - Full-time,DaouKiwoom Innovation,"Quận Bình Thạnh, Hồ Chí Minh",topdev,https://topdev.vn/detail-jobs/nhan-vien-che-ba...,None,None,OTHER,None,...,None,"[""Photoshop""]",None,"[""Competitive salary, salary review once a yea...",None,NaN,1.0,2026-01-11 15:12:08,2026-01-17 17:16:15,2026-01-11 15:12:08
4,3466,Webtoon Colorist - Họa sĩ tô màu - Full-time,DaouKiwoom Innovation,"Quận Bình Thạnh, Hồ Chí Minh",topdev,https://topdev.vn/detail-jobs/webtoon-colorist...,None,None,OTHER,None,...,None,None,None,"[""Competitive salary, salary review once a yea...",None,NaN,1.0,2026-01-11 15:12:11,2026-01-17 17:16:15,2026-01-11 15:12:11


## 7. Cấu hình trường cần dịch (Auto-detect tất cả các trường)

In [72]:
# Fields to KEEP ORIGINAL (do NOT translate these)
KEEP_ORIGINAL = ['title', 'location', 'company_name']

# System fields (do not translate)
SYSTEM_FIELDS = [
    'id', 'created_at', 'updated_at', 'crawled_at', 'posted_at',
    'is_active', 'source', 'source_url', 'job_id',
    'salary_min', 'salary_max', 'salary_currency',
    'experience_years_min', 'experience_years_max',
    'views_count', 'applications_count'
]

# Get all columns from dataframe
all_columns = df.columns.tolist()

# Auto-detect text fields (string/object types, not in protected lists)
TEXT_FIELDS = []
LIST_FIELDS = []

for col in all_columns:
    # Skip if in protected lists
    if col in KEEP_ORIGINAL or col in SYSTEM_FIELDS:
        continue
    
    # Check data type
    dtype = str(df[col].dtype)
    
    # Check if it's a text field (object/string type)
    if dtype == 'object':
        # Check if it contains JSON arrays (list fields)
        sample_value = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
        if sample_value:
            sample_str = str(sample_value).strip()
            # Detect JSON array by checking if it starts with [
            if sample_str.startswith('[') and sample_str.endswith(']'):
                try:
                    json.loads(sample_str)
                    LIST_FIELDS.append(col)
                    continue
                except:
                    pass
        
        # Otherwise it's a text field
        TEXT_FIELDS.append(col)

print('🔍 AUTO-DETECTED FIELDS TO TRANSLATE:')
print('='*100)
print(f'\n📝 Text fields ({len(TEXT_FIELDS)}):')
for field in TEXT_FIELDS:
    sample = str(df[field].dropna().iloc[0])[:60] if len(df[field].dropna()) > 0 else 'NULL'
    print(f'  • {field:<30} - Sample: {sample}')

print(f'\n📋 List/Array fields ({len(LIST_FIELDS)}):')
for field in LIST_FIELDS:
    sample = str(df[field].dropna().iloc[0])[:60] if len(df[field].dropna()) > 0 else 'NULL'
    print(f'  • {field:<30} - Sample: {sample}')

print(f'\n🔒 PROTECTED fields (will NOT translate - {len(KEEP_ORIGINAL)}):')
for field in KEEP_ORIGINAL:
    if field in df.columns:
        sample = str(df[field].dropna().iloc[0])[:60] if len(df[field].dropna()) > 0 else 'NULL'
        print(f'  • {field:<30} - Sample: {sample}')

print(f'\n⚙️  System fields (skipped - {len([f for f in SYSTEM_FIELDS if f in all_columns])}):')
for field in SYSTEM_FIELDS:
    if field in df.columns:
        print(f'  • {field}')

print('\n' + '='*100)
print(f'✅ Total fields to translate: {len(TEXT_FIELDS) + len(LIST_FIELDS)}')
print(f'🔒 Total protected fields: {len(KEEP_ORIGINAL)}')
print(f'⚙️  Total system fields: {len([f for f in SYSTEM_FIELDS if f in all_columns])}')

🔍 AUTO-DETECTED FIELDS TO TRANSLATE:

📝 Text fields (7):
  • description                    - Sample: Your role & responsibilities:
We’re looking for a
 front end
  • requirements_text              - Sample: Your skills & qualifications:

Experience in front end devel
  • job_type                       - Sample: OTHER
  • level                          - Sample: NULL
  • salary_text                    - Sample: Negotiable
  • education_level                - Sample: NULL
  • company_rating                 - Sample: NULL

📋 List/Array fields (4):
  • certifications                 - Sample: []
  • required_skills                - Sample: ["CSS", "HTML", "Git", "Restful Api", "SASS", "TypeScript", 
  • preferred_skills               - Sample: []
  • benefits                       - Sample: ["13th month bonusParticipate in large-scale and diverse pro

🔒 PROTECTED fields (will NOT translate - 3):
  • title                          - Sample: Frontend developer (PA project)
  • location     

## 8. Dịch dữ liệu (20-40 phút)

In [73]:
translator = TranslationHelper()
df_translated = df.copy()

# Verify protected fields are not in translation lists
for field in KEEP_ORIGINAL:
    if field in TEXT_FIELDS or field in LIST_FIELDS:
        print(f'⚠️ WARNING: {field} is marked for translation but should be protected!')
        TEXT_FIELDS = [f for f in TEXT_FIELDS if f != field]
        LIST_FIELDS = [f for f in LIST_FIELDS if f != field]
        print(f'   → Removed {field} from translation list')

print(f'🚀 Starting MAXIMUM AGGRESSIVE translation of {len(df):,} jobs')
print(f'⏱️  Estimated time: {len(df)*2/60:.0f} minutes')
print(f'🔒 Protected fields (will NOT translate): {", ".join(KEEP_ORIGINAL)}')
print(f'📝 Will translate: {len(TEXT_FIELDS)} text fields + {len(LIST_FIELDS)} list fields')
print(f'🔥 Mode: Translate EVERYTHING - NO SKIPPING, NO FILTERING')
print(f'📊 Total jobs to scan: {len(df):,}\n')

start_time = time.time()
translated_fields_count = 0
translated_jobs_count = 0
jobs_scanned = 0

for idx in tqdm(range(len(df_translated)), desc=f'Scanning ALL {len(df):,} jobs'):
    row = df_translated.iloc[idx]
    jobs_scanned += 1
    job_changed = False
    
    # Translate ALL text fields - NO FILTERING
    for field in TEXT_FIELDS:
        if field in df_translated.columns and pd.notna(row[field]):
            original_value = str(row[field])
            # TRANSLATE EVERYTHING - remove length check
            if original_value.strip():  # Only skip completely empty
                translated_value = translator.translate_text(original_value, field)
                # ALWAYS update, even if unchanged (force translation)
                df_translated.at[idx, field] = translated_value
                translated_fields_count += 1
                if translated_value != original_value:
                    job_changed = True
    
    # Translate ALL list fields (JSON arrays) - NO FILTERING
    for field in LIST_FIELDS:
        if field in df_translated.columns and pd.notna(row[field]):
            try:
                # Parse JSON array
                items = json.loads(row[field]) if isinstance(row[field], str) else row[field]
                if isinstance(items, list) and len(items) > 0:
                    # Translate ALL items in list
                    translated_items = translator.translate_list(items, field)
                    # ALWAYS update JSON
                    df_translated.at[idx, field] = json.dumps(translated_items, ensure_ascii=False)
                    translated_fields_count += len(items)
                    if translated_items != items:
                        job_changed = True
            except Exception as e:
                if idx < 5:  # Show first 5 errors only
                    print(f'\n⚠️ Error parsing {field} at job {idx}: {str(e)[:80]}')
    
    # Verify protected fields are NEVER changed
    for field in KEEP_ORIGINAL:
        if field in df.columns:
            original = df.at[idx, field]
            current = df_translated.at[idx, field]
            # Compare as strings, handling NaN
            orig_str = str(original) if pd.notna(original) else ''
            curr_str = str(current) if pd.notna(current) else ''
            if orig_str != curr_str:
                print(f'\n⚠️ ERROR: Protected field {field} changed at job {idx}!')
                print(f'   Original: {orig_str[:50]}')
                print(f'   Current:  {curr_str[:50]}')
                df_translated.at[idx, field] = original  # Restore original
    
    if job_changed:
        translated_jobs_count += 1
    
    # Progress reporting every 100 jobs
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df_translated) - idx - 1) / rate
        print(f'\n[{idx+1}/{len(df_translated)}] Time: {elapsed/60:.1f}m | '
              f'Remaining: {remaining/60:.1f}m | Rate: {rate:.1f} jobs/s')
        print(f'   Scanned: {jobs_scanned:,} | Vietnamese translated: {translated_jobs_count:,} | Fields: {translated_fields_count:,}')
        translator.print_stats()

elapsed_time = time.time() - start_time
print(f'\n✅ Translation complete!')
print(f'⏱️  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)')
print(f'⚡ Average rate: {len(df)/elapsed_time:.2f} jobs/second')
print(f'\n📊 DETAILED RESULTS:')
print(f'  ✅ Total jobs SCANNED: {jobs_scanned:,} (ALL jobs in database)')
print(f'  🇻🇳 Jobs with Vietnamese content: {translated_jobs_count:,}')
print(f'  🇬🇧 Jobs already in English: {jobs_scanned - translated_jobs_count:,}')
print(f'  📝 Total fields translated: {translated_fields_count:,}\n')
translator.print_stats()

# Final verification - check EVERY job
print(f'\n🔍 Final verification of protected fields:')
for field in KEEP_ORIGINAL:
    if field in df.columns:
        # Compare as strings to handle NaN properly
        orig_strs = df[field].fillna('').astype(str)
        trans_strs = df_translated[field].fillna('').astype(str)
        unchanged = (orig_strs == trans_strs).sum()
        if unchanged == len(df):
            print(f'  ✅ {field}: All {len(df):,} values unchanged')
        else:
            changed_count = len(df) - unchanged
            print(f'  ❌ {field}: WARNING - {changed_count} values changed!')
            # Show first 3 changed values
            shown = 0
            for i in range(len(df)):
                if orig_strs.iloc[i] != trans_strs.iloc[i]:
                    print(f'     Job {i}: "{orig_strs.iloc[i][:40]}" → "{trans_strs.iloc[i][:40]}"')
                    shown += 1
                    if shown >= 3:
                        break

# Show comprehensive summary
print(f'\n📊 FINAL SUMMARY:')
print(f'  Total jobs in database: {len(df):,}')
print(f'  Jobs scanned: {jobs_scanned:,} (100% of all jobs)')
print(f'  Jobs with Vietnamese translated: {translated_jobs_count:,} ({translated_jobs_count/len(df)*100:.1f}%)')
print(f'  Jobs already in English: {jobs_scanned - translated_jobs_count:,} ({(jobs_scanned - translated_jobs_count)/len(df)*100:.1f}%)')
print(f'  Protected fields: {", ".join(KEEP_ORIGINAL)} (NEVER translated)')
print(f'\n  ✅ ALL {len(df):,} JOBS WERE SCANNED AND PROCESSED!')

🚀 Starting MAXIMUM AGGRESSIVE translation of 2,985 jobs
⏱️  Estimated time: 100 minutes
🔒 Protected fields (will NOT translate): title, location, company_name
📝 Will translate: 7 text fields + 4 list fields
🔥 Mode: Translate EVERYTHING - NO SKIPPING, NO FILTERING
📊 Total jobs to scan: 2,985



Scanning ALL 2,985 jobs:   0%|          | 0/2985 [00:00<?, ?it/s]


[100/2985] Time: 1.3m | Remaining: 36.8m | Rate: 1.3 jobs/s
   Scanned: 100 | Vietnamese translated: 0 | Fields: 1,689
📊 Translated: 0 | Already English: 502 | Cached: 1,168 | Skipped: 19 | Errors: 0

[200/2985] Time: 2.2m | Remaining: 30.7m | Rate: 1.5 jobs/s
   Scanned: 200 | Vietnamese translated: 0 | Fields: 2,852
📊 Translated: 0 | Already English: 912 | Cached: 1,899 | Skipped: 41 | Errors: 0

[300/2985] Time: 3.2m | Remaining: 28.4m | Rate: 1.6 jobs/s
   Scanned: 300 | Vietnamese translated: 0 | Fields: 4,189
📊 Translated: 0 | Already English: 1,355 | Cached: 2,789 | Skipped: 45 | Errors: 0

[400/2985] Time: 5.2m | Remaining: 33.5m | Rate: 1.3 jobs/s
   Scanned: 400 | Vietnamese translated: 0 | Fields: 7,835
📊 Translated: 0 | Already English: 2,255 | Cached: 5,419 | Skipped: 161 | Errors: 0

[500/2985] Time: 7.2m | Remaining: 35.6m | Rate: 1.2 jobs/s
   Scanned: 500 | Vietnamese translated: 0 | Fields: 11,403
📊 Translated: 0 | Already English: 3,149 | Cached: 7,976 | Skipped: 27

## 9. Xem trước kết quả

In [74]:
print('🔍 TRANSLATION COMPARISON (Before → After)\n')
print('='*100)

# Show first job
idx = 0
job = df.iloc[idx]
print(f'Job ID: {job["id"]} | Title: {job["title"]}')
print(f'Company: {job["company_name"]} | Location: {job["location"]}')
print('='*100)

# Compare translated fields
for field in TEXT_FIELDS:
    if pd.notna(job[field]):
        orig = str(df.iloc[idx][field])[:200]
        trans = str(df_translated.iloc[idx][field])[:200]
        if orig != trans:
            print(f'\n📝 {field.upper()}:')
            print(f'  Before: {orig}...')
            print(f'  After:  {trans}...')

# Compare list fields
for field in LIST_FIELDS:
    if pd.notna(job[field]):
        try:
            orig_list = json.loads(str(df.iloc[idx][field])) if isinstance(df.iloc[idx][field], str) else df.iloc[idx][field]
            trans_list = json.loads(str(df_translated.iloc[idx][field])) if isinstance(df_translated.iloc[idx][field], str) else df_translated.iloc[idx][field]
            if orig_list != trans_list and isinstance(orig_list, list) and isinstance(trans_list, list):
                print(f'\n📝 {field.upper()}:')
                print(f'  Before: {orig_list[:3]}...')
                print(f'  After:  {trans_list[:3]}...')
        except:
            pass

# Verify protected fields
print(f'\n\n🔒 PROTECTED FIELDS (must be unchanged):')
print('='*100)
for field in KEEP_ORIGINAL:
    if field in df.columns:
        orig = str(df.iloc[idx][field])
        trans = str(df_translated.iloc[idx][field])
        status = '✅ UNCHANGED' if orig == trans else '❌ CHANGED'
        print(f'{field}: {orig} → {trans} [{status}]')

print('\n' + '='*100)
print(f'✅ Sample comparison complete')

🔍 TRANSLATION COMPARISON (Before → After)

Job ID: 3462 | Title: Frontend developer (PA project)
Company: CUBICASA | Location: Quận 1, Hồ Chí Minh


🔒 PROTECTED FIELDS (must be unchanged):
title: Frontend developer (PA project) → Frontend developer (PA project) [✅ UNCHANGED]
location: Quận 1, Hồ Chí Minh → Quận 1, Hồ Chí Minh [✅ UNCHANGED]
company_name: CUBICASA → CUBICASA [✅ UNCHANGED]

✅ Sample comparison complete


## 10. Lưu backup CSV

In [75]:
from datetime import datetime
backup_file = f'jobs_translated_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_translated.to_csv(backup_file, index=False, encoding='utf-8-sig')
print(f'✅ Backup: {backup_file}')

try:
    from google.colab import files
    files.download(backup_file)
except:
    print('💾 File saved locally')

✅ Backup: jobs_translated_20260117_191204.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Cập nhật Database

In [76]:
confirm = input('⚠️  Update database with translated data? (yes/no): ')

if confirm.lower() == 'yes':
    print('\n🔄 Updating database...')
    print('⏰ This will update the "updated_at" timestamp for all modified rows')
    print('🔥 Updating ALL jobs (active and inactive)\n')
    
    update_count = 0
    error_count = 0
    
    # Use transaction for better performance
    from sqlalchemy import Connection
    
    with engine.begin() as conn:  # Auto-commit transaction
        for idx in tqdm(range(len(df_translated)), desc='Updating ALL jobs in database'):
            row = df_translated.iloc[idx]
            try:
                # Build dynamic UPDATE query for all translated fields
                update_fields = []
                params = {'id': int(row['id'])}
                
                # Add text fields to update
                for field in TEXT_FIELDS:
                    if field in df_translated.columns:
                        update_fields.append(f'`{field}` = :{field}')
                        params[field] = row[field] if pd.notna(row[field]) else None
                
                # Add list fields to update
                for field in LIST_FIELDS:
                    if field in df_translated.columns:
                        update_fields.append(f'`{field}` = :{field}')
                        params[field] = row[field] if pd.notna(row[field]) else None
                
                # Execute update with updated_at = NOW()
                if update_fields:
                    update_query = f'''
                        UPDATE jobs SET
                            {", ".join(update_fields)},
                            `updated_at` = NOW()
                        WHERE id = :id
                    '''
                    conn.execute(text(update_query), params)
                    update_count += 1
                    
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f'\n⚠️ Error updating job {row["id"]}: {str(e)[:100]}')
                    print(f'   Field causing error: Check if all TEXT_FIELDS and LIST_FIELDS exist')
    
    print(f'\n✅ Updated: {update_count:,} jobs in database')
    print(f'❌ Errors: {error_count:,}')
    
    if error_count == 0:
        # Verify protected fields unchanged in database (check ALL jobs now)
        print(f'\n🔍 Verifying protected fields in database...')
        verify_query = f'SELECT id, `{("`, `").join(KEEP_ORIGINAL)}` FROM jobs ORDER BY id'
        df_verify = pd.read_sql(verify_query, engine)
        
        all_match = True
        for field in KEEP_ORIGINAL:
            if field in df.columns and field in df_verify.columns:
                matches = (df[field].fillna('') == df_verify[field].fillna('')).sum()
                if matches == len(df):
                    print(f'  ✅ {field}: All {len(df):,} values unchanged')
                else:
                    print(f'  ❌ {field}: {len(df)-matches} values changed!')
                    all_match = False
        
        # Verify updated_at was changed
        print(f'\n🕐 Verifying updated_at timestamps...')
        check_query = 'SELECT COUNT(*) as updated_count FROM jobs WHERE updated_at >= DATE_SUB(NOW(), INTERVAL 5 MINUTE)'
        result = pd.read_sql(check_query, engine)
        recently_updated = result['updated_count'].iloc[0]
        print(f'  ✅ {recently_updated:,} jobs have updated_at within last 5 minutes')
        
        if all_match:
            print(f'\n✅ SUCCESS: All protected fields verified unchanged in database')
            print(f'✅ SUCCESS: updated_at timestamps updated correctly')
        else:
            print(f'\n⚠️ WARNING: Some protected fields were changed!')
    else:
        print(f'\n⚠️ WARNING: {error_count} errors occurred during update')
        print(f'   Recommendation: Check the errors above and verify field names')
        
else:
    print('❌ Update cancelled by user')


🔄 Updating database...
⏰ This will update the "updated_at" timestamp for all modified rows
🔥 Updating ALL jobs (active and inactive)



Updating ALL jobs in database:   0%|          | 0/2985 [00:00<?, ?it/s]


✅ Updated: 2,985 jobs in database
❌ Errors: 0

🔍 Verifying protected fields in database...
  ✅ title: All 2,985 values unchanged
  ✅ location: All 2,985 values unchanged
  ✅ company_name: All 2,985 values unchanged

🕐 Verifying updated_at timestamps...
  ✅ 1,348 jobs have updated_at within last 5 minutes

✅ SUCCESS: All protected fields verified unchanged in database
✅ SUCCESS: updated_at timestamps updated correctly


## 12. Cleanup và Tổng kết

In [77]:
# Cleanup temporary files
for f in temp_files:
    try: 
        os.unlink(f)
        print(f'🗑️  Deleted temp file: {f}')
    except: 
        pass

print('\n' + '='*80)
print('🎉 TRANSLATION COMPLETE - FINAL SUMMARY')
print('='*80)
print(f'📊 Total jobs processed: {len(df):,}')
print(f'💾 Backup file: {backup_file}')
print(f'⏱️  Processing time: {elapsed_time/60:.1f} minutes')
print()

# Translation statistics
summary = translator.get_summary()
print('📈 TRANSLATION STATISTICS:')
print(f'  ✅ Successfully translated: {summary["stats"]["translated"]:,}')
print(f'  💨 Cached (reused): {summary["stats"]["cached"]:,}')
print(f'  ⏭️  Skipped (already English): {summary["stats"]["skipped"]:,}')
print(f'  ❌ Errors: {summary["stats"]["errors"]:,}')
print(f'  🔒 Protected (restored): {summary["stats"]["protected"]:,}')
print(f'  📦 Cache size: {summary["cache_size"]:,} unique translations')
print()

print('📝 FIELDS TRANSLATED:')
print(f'  Text fields: {", ".join(TEXT_FIELDS)}')
print(f'  List fields: {", ".join(LIST_FIELDS)}')
print()

print('🔒 FIELDS KEPT ORIGINAL (NOT translated):')
print(f'  {", ".join(KEEP_ORIGINAL)}')
print()

# Final verification
all_verified = True
for field in KEEP_ORIGINAL:
    if field in df.columns:
        unchanged = (df[field] == df_translated[field]).sum()
        if unchanged != len(df):
            all_verified = False
            print(f'⚠️  WARNING: {field} has {len(df)-unchanged} changed values!')

if all_verified:
    print('✅ VERIFICATION PASSED: All protected fields unchanged')
else:
    print('❌ VERIFICATION FAILED: Some protected fields were modified')

print('='*80)

🗑️  Deleted temp file: /tmp/tmphix2r5z1.pem

🎉 TRANSLATION COMPLETE - FINAL SUMMARY
📊 Total jobs processed: 2,985
💾 Backup file: jobs_translated_20260117_191204.csv
⏱️  Processing time: 105.8 minutes

📈 TRANSLATION STATISTICS:
  ✅ Successfully translated: 8,194
  💨 Cached (reused): 42,173


KeyError: 'skipped'

## 13. Debug - Check Specific Job

In [ ]:
# Check specific job ID from database
job_id = 4521

print(f'🔍 Checking Job ID: {job_id}\n')
print('='*100)

# Get job from database
query = f'SELECT * FROM jobs WHERE id = {job_id}'
job_db = pd.read_sql(query, engine)

if len(job_db) == 0:
    print(f'❌ Job {job_id} not found in database!')
else:
    job = job_db.iloc[0]
    
    print(f'📋 Job ID: {job["id"]}')
    print(f'🏢 Company: {job["company_name"]}')
    print(f'📍 Location: {job["location"]}')
    print(f'💼 Title: {job["title"]}')
    print(f'🌐 Source: {job["source"]}')
    print('='*100)
    
    # Check all text and list fields for Vietnamese content
    print(f'\n🔍 Checking all fields for Vietnamese content:\n')
    
    # Check text fields
    for field in TEXT_FIELDS:
        if field in job_db.columns and pd.notna(job[field]):
            value = str(job[field])
            if value and len(value) > 10:
                # Show first 200 chars
                print(f'\n📝 {field.upper()}:')
                print(f'   {value[:200]}...')
                
                # Try to detect language
                try:
                    from langdetect import detect
                    lang = detect(value[:500])
                    if lang == 'vi':
                        print(f'   🇻🇳 DETECTED: Vietnamese content! This should be translated!')
                    else:
                        print(f'   Language: {lang}')
                except:
                    print(f'   ⚠️ Could not detect language')
    
    # Check list fields
    for field in LIST_FIELDS:
        if field in job_db.columns and pd.notna(job[field]):
            try:
                items = json.loads(job[field]) if isinstance(job[field], str) else job[field]
                if isinstance(items, list) and len(items) > 0:
                    print(f'\n📋 {field.upper()}: {items}')
                    # Check if any item is Vietnamese
                    for item in items[:3]:  # Check first 3 items
                        if isinstance(item, str) and len(item) > 5:
                            try:
                                lang = detect(item)
                                if lang == 'vi':
                                    print(f'   🇻🇳 DETECTED Vietnamese: "{item}"')
                            except:
                                pass
            except Exception as e:
                print(f'   ⚠️ Error parsing {field}: {str(e)[:50]}')
    
    print('\n' + '='*100)
    
    # Check if this job was in our df_translated
    if 'df_translated' in globals():
        job_in_df = df_translated[df_translated['id'] == job_id]
        if len(job_in_df) > 0:
            print(f'\n✅ Job {job_id} WAS in the translation batch')
            print(f'   It was processed during translation')
        else:
            print(f'\n⚠️ Job {job_id} was NOT in the translation batch!')
            print(f'   This job was not loaded from database (check WHERE clause)')
    
    # Check if job is active
    if pd.notna(job['is_active']):
        if job['is_active'] == 1 or job['is_active'] == True:
            print(f'   ✅ Job is_active = 1 (should be included)')
        else:
            print(f'   ❌ Job is_active = {job["is_active"]} (NOT included in WHERE is_active = 1)')
    
    # Check updated_at
    if pd.notna(job['updated_at']):
        print(f'   🕐 updated_at: {job["updated_at"]}')

🔍 Checking Job ID: 4521

📋 Job ID: 4521
🏢 Company: CÔNG TY CỔ PHẦN TOKYO TECH LAB VIỆT NAM
📍 Location: Tòa B2 Roman Plaza, Đường Tố Hữu Phường Đại Mỗ, Quận Nam Từ Liêm, Thành phố Hà Nội, Việt Nam
💼 Title: Senior Software Engineer(NodeJS) - 5 Năm Kinh Nghiệm - Thu Nhập Từ 40 - 55 Triệu/Tháng
🌐 Source: topcv

🔍 Checking all fields for Vietnamese content:


📝 DESCRIPTION:
   Vị trí Senior Software Engineer là vị trí quan trọng tại Tokyo Tech Lab, là người có chuyên môn về kỹ thuật để phát triển dự án vững mạnh cho công ty.Phát triển ứng dụng phần mềm cho khách hàng trong ...
   🇻🇳 DETECTED: Vietnamese content! This should be translated!

📝 REQUIREMENTS_TEXT:
   Tốt nghiệp đại học chuyên ngành công nghệ thông tin, có từ 5 năm kinh nghiệm ở các dự án phát triển phần mềm với vai trò kỹ sư / lập trình viên.Thành thạo một hoặc nhiều trong số những ngôn ngữ lập tr...
   🇻🇳 DETECTED: Vietnamese content! This should be translated!

📝 SALARY_TEXT:
   40 - 55 triệu...
   🇻🇳 DETECTED: Vietnamese con